In [1]:
"""
GPT-OSS-120B Joke Generator
Generates creative jokes from word pairs using OpenAI's GPT-OSS-120B API
"""

import os
from typing import List, Dict, Optional
from openai import OpenAI
import json

In [2]:
class GPT4JokeGenerator:
    """
    Generates creative jokes from word pairs using GPT-OSS-120B with optimized prompts.
    """

    def __init__(
        self,
        api_key: Optional[str] = None,
        temperature: float = 0.9,
        top_p: float = 1.0,
        model: str = "openai/gpt-oss-120b",
        max_retries: int = 1
    ):
        """
        Initialize the GPT-OSS-120B joke generator.

        Args:
            api_key: NVIDIA API key (if None, reads from NVIDIA_API_KEY env var)
            temperature: Sampling temperature (0.7-1.0 recommended for creativity)
            top_p: Nucleus sampling parameter (default: 1.0)
            model: NVIDIA model to use (default: openai/gpt-oss-120b)
            max_retries: Maximum number of retry attempts on failure
        """
        self.client = OpenAI(
            base_url="https://integrate.api.nvidia.com/v1",
            api_key=api_key or os.getenv("NVIDIA_API_KEY")
        )
        self.temperature = temperature
        self.top_p = top_p
        self.model = model
        self.max_retries = max_retries

        # System prompt - establishes the AI's role and guidelines
        self.system_prompt = """You are a world-class comedy writer specializing in creative, unexpected humor. Your expertise includes:

• Wordplay & Puns: Creating clever double meanings and linguistic twists
• Absurdist Humor: Finding comedy in unexpected juxtapositions
• Observational Comedy: Highlighting the amusing aspects of everyday scenarios
• Conceptual Blending: Merging unrelated concepts in surprising ways

Your jokes should be:
✓ Original and creative (avoid clichés and obvious connections)
✓ Concise (1-3 sentences maximum)
✓ Family-friendly (no offensive, crude, or inappropriate content)
✓ Surprising (the punchline should be unexpected yet logical in hindsight)
✓ Self-contained (understandable without additional context)
✓ Naturally incorporate both words in a meaningful way

Focus on creating jokes that make people think "I never would have connected those ideas, but it's brilliant!"

Your goal is to find the most creative, unexpected, yet logical connections between the two words."""

    def _build_user_prompt(self, word1: str, word2: str, num_jokes: int = 10) -> str:
        """
        Constructs the user prompt for joke generation.

        Args:
            word1: First word in the pair
            word2: Second word in the pair
            num_jokes: Number of jokes to generate

        Returns:
            Formatted prompt string
        """
        prompt = f"""Generate {num_jokes} creative, hilarious jokes that cleverly connect these two words:

Word 1: {word1}
Word 2: {word2}

Requirements:
1. Each joke MUST incorporate BOTH words naturally and meaningfully
2. Explore DIFFERENT comedic approaches across the {num_jokes} jokes:
   - Some should use wordplay or puns
   - Some should be absurdist or surreal
   - Some should be observational or relatable
   - Some should create unexpected conceptual blends
3. Avoid forced or obvious connections - find genuinely clever relationships
4. Make each joke distinct and unique from the others
5. Keep jokes concise (1-3 sentences each)
6. Ensure all jokes are appropriate and family-friendly

Think creatively about:
- What unexpected contexts could combine these words?
- What if one word had properties of the other?
- What professions, situations, or scenarios could involve both?
- What absurd scenarios would require both items?
- What puns or wordplay connect these concepts?

Format your response as a valid JSON object with a "jokes" key containing an array of strings:
{{"jokes": ["joke 1", "joke 2", "joke 3", ...]}}

Generate {num_jokes} unique, creative jokes now:"""

        return prompt

    def generate_jokes(
        self,
        word1: str,
        word2: str,
        num_jokes: int = 10,
        verbose: bool = False
    ) -> List[str]:
        """
        Generate jokes from a word pair using GPT-OSS-120B with streaming.

        Args:
            word1: First word in the pair
            word2: Second word in the pair
            num_jokes: Number of jokes to generate (default: 10)
            verbose: If True, print streaming output in real-time

        Returns:
            List of generated joke strings

        Raises:
            Exception: If generation fails after max_retries attempts
        """
        user_prompt = self._build_user_prompt(word1, word2, num_jokes)

        for attempt in range(self.max_retries):
            try:
                # Use streaming completion
                completion = self.client.chat.completions.create(
                    model=self.model,
                    messages=[
                        {"role": "system", "content": self.system_prompt},
                        {"role": "user", "content": user_prompt}
                    ],
                    temperature=self.temperature,
                    top_p=self.top_p,
                    max_tokens=4096,
                    stream=True
                )

                # Collect streamed response
                full_content = ""
                reasoning_content = ""

                if verbose:
                    print("\n🤖 Model generating jokes (streaming)...\n")

                for chunk in completion:
                    if not getattr(chunk, "choices", None):
                        continue

                    # Handle reasoning content if present
                    reasoning = getattr(chunk.choices[0].delta, "reasoning_content", None)
                    if reasoning:
                        reasoning_content += reasoning
                        if verbose:
                            print(reasoning, end="", flush=True)

                    # Handle main content
                    if chunk.choices and chunk.choices[0].delta.content is not None:
                        content_chunk = chunk.choices[0].delta.content
                        full_content += content_chunk
                        if verbose:
                            print(content_chunk, end="", flush=True)

                if verbose:
                    print("\n\n✓ Generation complete, parsing response...\n")

                # Parse the complete response
                jokes = self._parse_response(full_content, num_jokes)

                # Validate we got enough jokes
                if len(jokes) >= int(num_jokes * 0.7):
                    return jokes[:num_jokes]
                else:
                    if attempt < self.max_retries - 1:
                        print(f"Attempt {attempt + 1}/{self.max_retries}: Only got {len(jokes)}/{num_jokes} jokes, retrying...")

            except json.JSONDecodeError as e:
                print(f"Attempt {attempt + 1}/{self.max_retries}: JSON parsing error - {e}")
                if attempt == self.max_retries - 1:
                    # Last attempt - try fallback extraction
                    raise Exception(f"Failed to extract jokes!")

            except Exception as e:
                print(f"Attempt {attempt + 1}/{self.max_retries}: Error - {e}")
                if attempt == self.max_retries - 1:
                    raise Exception(f"Failed to generate jokes after {self.max_retries} attempts: {e}")

        raise Exception(f"Failed to generate {num_jokes} jokes after {self.max_retries} attempts")

    def _parse_response(self, content: str, num_jokes: int) -> List[str]:
        """
        Parse the model response to extract jokes.
        Tries JSON parsing first, then falls back to text extraction.

        Args:
            content: Raw response content from the model
            num_jokes: Target number of jokes

        Returns:
            List of joke strings
        """
        # Try JSON parsing first
        try:
            # Look for JSON in the content
            json_start = content.find('{')
            json_end = content.rfind('}') + 1

            if json_start != -1 and json_end > json_start:
                json_str = content[json_start:json_end]
                jokes_data = json.loads(json_str)
                jokes = self._extract_jokes_from_json(jokes_data)

                # Clean and validate
                jokes = [str(joke).strip() for joke in jokes if joke and len(str(joke).strip()) > 20]

                if len(jokes) > 0:
                    return jokes

        except (json.JSONDecodeError, ValueError):
            raise Exception(f"Failed to extract jokes!")

        print('Failed to extract jokes!')

    def _extract_jokes_from_json(self, jokes_data: dict) -> List[str]:
        """
        Extract jokes list from various JSON response formats.

        Args:
            jokes_data: Parsed JSON data from API response

        Returns:
            List of joke strings
        """
        # If the response is already a list
        if isinstance(jokes_data, list):
            return jokes_data

        # If it's a dict, try common key names
        if isinstance(jokes_data, dict):
            for key in ['jokes', 'responses', 'results', 'output', 'data', 'items']:
                if key in jokes_data and isinstance(jokes_data[key], list):
                    return jokes_data[key]

            # If no standard key found, try to find any list value
            for value in jokes_data.values():
                if isinstance(value, list) and len(value) > 0:
                    return value

        raise ValueError("Could not extract jokes list from JSON response")

    def generate_with_metadata(
        self,
        word1: str,
        word2: str,
        num_jokes: int = 10,
        verbose: bool = False
    ) -> Dict:
        """
        Generate jokes and return with generation metadata.

        Args:
            word1: First word in the pair
            word2: Second word in the pair
            num_jokes: Number of jokes to generate
            verbose: If True, print streaming output in real-time

        Returns:
            Dictionary containing:
                - jokes: List of generated joke strings
                - metadata: Dict with generation parameters and stats
        """
        jokes = self.generate_jokes(word1, word2, num_jokes, verbose=verbose)

        return {
            "jokes": jokes,
            "metadata": {
                "word_pair": [word1, word2],
                "model": self.model,
                "temperature": self.temperature,
                "top_p": self.top_p,
                "num_requested": num_jokes,
                "num_generated": len(jokes),
                "generation_successful": len(jokes) >= int(num_jokes * 0.7)
            }
        }

    def batch_generate(
        self,
        word_pairs: List[tuple],
        num_jokes_per_pair: int = 10
    ) -> Dict[str, Dict]:
        """
        Generate jokes for multiple word pairs.

        Args:
            word_pairs: List of (word1, word2) tuples
            num_jokes_per_pair: Number of jokes to generate per pair

        Returns:
            Dictionary mapping "word1_word2" to generation results
        """
        results = {}

        for word1, word2 in word_pairs:
            key = f"{word1}_{word2}"
            try:
                results[key] = self.generate_with_metadata(word1, word2, num_jokes_per_pair)
                results[key]["status"] = "success"
            except Exception as e:
                results[key] = {
                    "jokes": [],
                    "metadata": {
                        "word_pair": [word1, word2],
                        "error": str(e)
                    },
                    "status": "failed"
                }

        return results

In [3]:
"""
Example usage and testing of the GPT-OSS-120B Joke Generator.
"""
print("=" * 80)
print("GPT-OSS-120B JOKE GENERATOR (NVIDIA)")
print("=" * 80)

# Initialize generator
generator = GPT4JokeGenerator(
    temperature=0.9,
    top_p=1.0,
    max_retries=1,
    api_key='REDACTED_CREDENTIAL'
)

# Example 1: Single word pair
print("\n" + "=" * 80)
print("EXAMPLE 1: Basic Generation")
print("=" * 80)

word1, word2 = "telescope", "sandwich"
print(f"\n🎯 Generating jokes for: '{word1}' + '{word2}'")
print("-" * 80)

try:
    result = generator.generate_with_metadata(word1, word2, num_jokes=10, verbose=False)

    print(f"\n✓ Successfully generated {len(result['jokes'])} jokes:\n")

    for i, joke in enumerate(result['jokes'], 1):
        print(f"{i:2d}. {joke}\n")

    print("\n📊 Generation Metadata:")
    print(f"   Model: {result['metadata']['model']}")
    print(f"   Temperature: {result['metadata']['temperature']}")
    print(f"   Top-p: {result['metadata']['top_p']}")
    print(f"   Requested: {result['metadata']['num_requested']}")
    print(f"   Generated: {result['metadata']['num_generated']}")
    print(f"   Success: {result['metadata']['generation_successful']}")

except Exception as e:
    print(f"\n❌ Error: {e}")
    print("\nMake sure to set your NVIDIA_API_KEY environment variable:")
    print("  export NVIDIA_API_KEY='your-key-here'")

GPT-OSS-120B JOKE GENERATOR (NVIDIA)

EXAMPLE 1: Basic Generation

🎯 Generating jokes for: 'telescope' + 'sandwich'
--------------------------------------------------------------------------------

✓ Successfully generated 10 jokes:

 1. I aimed my telescope at my lunch, hoping to spot the cheese, and discovered the sandwich was already a million miles away—turns out my meal had better optics than I do.

 2. The only thing more layered than a telescope’s lenses is a triple‑decker sandwich, which is why astronomers call it the “culinary event horizon.”

 3. My kid tried to eat a sandwich through a telescope, claiming it would make the bite “extra‑cosmic”; the only thing that got magnified was the mess on the floor.

 4. At the space museum, the “Astronaut’s Lunch” exhibit features a telescope turned into a sandwich holder—so you can admire your sub while it orbits your plate.

 5. I once ordered a “telescope sandwich” at a deli: it came with extra slices of bread so thick you needed a m

In [1]:
# inside gpt_joke_generation.py
from prompt_pipeline import HumorPromptPipeline, PipelineConfig, LLMClient
from patterns import JOKE_PATTERNS  # :contentReference[oaicite:14]{index=14}

class NvidiaChatClient(LLMClient):
    def __init__(self, client, model: str, system_prompt: str):
        self.client = client
        self.model = model
        self.system_prompt = system_prompt

    def complete(self, prompt: str, *, temperature: float, max_tokens: int) -> str:
        # non-streaming is easier for “step1/step2”
        resp = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": prompt},
            ],
            temperature=temperature,
            top_p=1.0,
            max_tokens=max_tokens,
            stream=False,
        )
        return resp.choices[0].message.content


class GPT4JokeGenerator:
    # keep your __init__ as-is :contentReference[oaicite:15]{index=15}

    def generate_jokes_with_pipeline(self, word1: str, word2: str, num_jokes: int = 10) -> list[str]:
        text = f"{word1} {word2}"  # pipeline already supports “two words” input :contentReference[oaicite:16]{index=16}
        llm = NvidiaChatClient(self.client, self.model, self.system_prompt)
        pipe = HumorPromptPipeline(llm, PipelineConfig())

        jokes = []
        # Option A: random patterns
        import random
        for _ in range(num_jokes):
            pattern = random.choice(JOKE_PATTERNS)
            jokes.append(pipe.run(pattern=pattern, text=text))
        return jokes

    def generate_one_per_pattern(self, word1: str, word2: str) -> dict[str, str]:
        text = f"{word1} {word2}"
        llm = NvidiaChatClient(self.client, self.model, self.system_prompt)
        pipe = HumorPromptPipeline(llm, PipelineConfig())

        out = {}
        for pattern in JOKE_PATTERNS:
            out[pattern] = pipe.run(pattern=pattern, text=text)
        return out
